# 00 · Kurulum ve Temel Kavramlar
Bu serideki tüm notebook'lar kendi başına çalışır. Bu notebook ortam kurulumunu ve makine öğrenmesinin temel kavramlarını anlatır.

| # | Notebook | Konu |
|---|---|---|
| 00 | Kurulum ve Temel Kavramlar | Ortam, kavramlar, metrikler |
| 01 | Çoklu Lineer Regresyon | Ayrıntılı teori + 4 farklı kodlama yolu |
| 02 | Regresyon Modelleri | Ridge, Lasso, ElasticNet, Polinom, Ağaçlar, SVR, KNN |
| 03 | Sınıflandırma Modelleri | Lojistik, KNN, SVM, Naive Bayes, Karar Ağacı |
| 04 | Ensemble Modeller | Random Forest, Boosting, XGBoost, LightGBM, Stacking |
| 05 | Kümeleme | K-Means, DBSCAN, Hiyerarşik |
| 06 | Boyut İndirgeme | PCA, t-SNE |
| 07 | Anomali Tespiti | Isolation Forest, LOF |
| 08 | Zaman Serisi | ARIMA / SARIMA |
| 09 | Yapay Sinir Ağları | MLP, Keras |
| 10 | Model Seçimi | Cross-validation, GridSearch, kaydetme |
| 11 | Model Seçim Rehberi | Hangi model ne zaman? |

## 1. Kurulum
Terminalde (PowerShell) proje klasöründe sanal ortam oluşturun:
```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r ML_Notebooks\requirements.txt
```
Ardından VS Code / Jupyter'da kernel olarak `.venv` ortamını seçin.

In [ ]:
# Kütüphanelerin kurulu olup olmadığını kontrol edelim
import importlib
paketler = ["numpy", "pandas", "matplotlib", "seaborn", "sklearn", "scipy",
            "statsmodels", "xgboost", "lightgbm", "tensorflow", "joblib"]
for p in paketler:
    try:
        m = importlib.import_module(p)
        print(f"✓ {p:12s} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"✗ {p:12s} KURULU DEĞİL")

> Tüm satırlar ✓ ise ortam hazır. ✗ olan paketleri `pip install <paket>` ile kurun (`sklearn` için paket adı `scikit-learn`).

## 2. Temel Kavramlar

| Kavram | Açıklama |
|---|---|
| **Özellik (feature, X)** | Modelin girdi olarak kullandığı değişkenler (m², oda sayısı…) |
| **Hedef (target, y)** | Tahmin edilmek istenen değer (fiyat, hastalık var/yok…) |
| **Gözetimli öğrenme** | Etiketli veriyle öğrenme → *Regresyon* (sürekli y) ve *Sınıflandırma* (kategorik y) |
| **Gözetimsiz öğrenme** | Etiketsiz veride yapı bulma → *Kümeleme*, *Boyut indirgeme* |
| **Eğitim / Test ayrımı** | Model eğitim verisiyle öğrenir, hiç görmediği test verisiyle değerlendirilir |
| **Aşırı öğrenme (overfitting)** | Eğitimde çok iyi, testte kötü → model ezberliyor |
| **Eksik öğrenme (underfitting)** | Hem eğitimde hem testte kötü → model çok basit |
| **Düzenlileştirme (regularization)** | Karmaşıklığı cezalandırarak overfitting'i azaltma |

### İş akışı
Veri → Keşifsel Analiz → Eksik/Aykırı Değer → **Train/Test Ayır** → Ön İşleme (sadece train'e fit) → Model → Değerlendirme

> ⚠️ **Altın kural:** Veriyi **önce** train/test olarak ayırın, ölçekleyici/encoder gibi ön işlemleri **sadece train verisine `fit`** edin. Aksi halde test bilgisi modele sızar (*data leakage*). En kolay yol `Pipeline` kullanmaktır.

## 3. Overfitting / Underfitting'i görelim
Aynı veriye 1., 4. ve 15. derece polinom uyduralım.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 1, 40))
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.2, 40)
x_tr, x_te, y_tr, y_te = train_test_split(x.reshape(-1, 1), y, test_size=0.3, random_state=0)

grid = np.linspace(0, 1, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, derece, baslik in zip(axes, [1, 4, 15], ["Underfitting", "İyi Uyum", "Overfitting"]):
    m = make_pipeline(PolynomialFeatures(derece), LinearRegression()).fit(x_tr, y_tr)
    tr_mse = mean_squared_error(y_tr, m.predict(x_tr))
    te_mse = mean_squared_error(y_te, m.predict(x_te))
    ax.scatter(x_tr, y_tr, c="tab:blue", label="Train")
    ax.scatter(x_te, y_te, c="tab:orange", label="Test")
    ax.plot(grid, m.predict(grid), c="tab:green", lw=2)
    ax.set(title=f"{baslik} (derece={derece})\nTrain MSE={tr_mse:.3f} | Test MSE={te_mse:.3f}", ylim=(-2, 2))
    ax.legend()
plt.tight_layout(); plt.show()

**Yorum:**
- **Derece 1:** Hem train hem test hatası yüksek → model çok basit (underfitting).
- **Derece 4:** İki hata da düşük ve birbirine yakın → ideal.
- **Derece 15:** Train hatası çok düşük ama test hatası patlar → ezberleme (overfitting).

## 4. Değerlendirme Metrikleri

| Regresyon | Sınıflandırma |
|---|---|
| **MAE** – ortalama mutlak hata | **Accuracy** – doğruluk |
| **MSE / RMSE** – (kök) ortalama kare hata | **Precision** – "pozitif dediklerimin ne kadarı doğru?" |
| **R²** – açıklanan varyans oranı (1 = mükemmel) | **Recall** – "gerçek pozitiflerin ne kadarını yakaladım?" |
| **MAPE** – yüzde hata | **F1** – precision & recall dengesi, **ROC-AUC** |

In [ ]:
from sklearn.metrics import (mean_absolute_error, r2_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix)

# Regresyon metrikleri – küçük örnek
y_gercek = np.array([100, 150, 200, 250, 300])
y_tahmin = np.array([110, 140, 210, 240, 330])
print("REGRESYON")
print(f"  MAE : {mean_absolute_error(y_gercek, y_tahmin):.1f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_gercek, y_tahmin)):.1f}")
print(f"  R²  : {r2_score(y_gercek, y_tahmin):.3f}")

# Sınıflandırma metrikleri – küçük örnek
c_gercek = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 1])
c_tahmin = np.array([1, 0, 0, 1, 0, 1, 1, 0, 1, 0])
print("\nSINIFLANDIRMA")
print(f"  Accuracy : {accuracy_score(c_gercek, c_tahmin):.2f}")
print(f"  Precision: {precision_score(c_gercek, c_tahmin):.2f}")
print(f"  Recall   : {recall_score(c_gercek, c_tahmin):.2f}")
print(f"  F1       : {f1_score(c_gercek, c_tahmin):.2f}")
print("  Confusion matrix:\n", confusion_matrix(c_gercek, c_tahmin))

**Yorum:** Confusion matrix'te satırlar gerçek, sütunlar tahmin edilen sınıfı gösterir. Örnekte 6 gerçek pozitiften 4'ü yakalandı (recall ≈ 0.67), pozitif denen 5 örnekten 4'ü doğru (precision = 0.80).

> Sınıflar dengesizse (örn. %99 normal, %1 dolandırıcılık) **accuracy yanıltıcıdır**; F1, Recall, PR-AUC kullanın.

## Özet
### Data Analysis Key Findings
- Model karmaşıklığı arttıkça train hatası hep düşer, ama test hatası önce düşer sonra yükselir — doğru karmaşıklık test/validation hatasıyla seçilir.
- Ön işleme adımları yalnızca train verisine fit edilmelidir; `Pipeline` bunu otomatik garanti eder.
### Insights or Next Steps
- Sıradaki notebook: **01_Coklu_Lineer_Regresyon**.